In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from xgboost import XGBClassifier
from pathlib import Path

DATA_PATH = Path("data/raw/online_retail.xlsx")

st.set_page_config(page_title="NeuralRetail Dashboard", layout="wide")

st.title("📊 NeuralRetail - AI Retail Analytics Dashboard")

# -------------------------------
# LOAD DATA
# -------------------------------
@st.cache_data
def load_data(path: Path):
    if not path.exists():
        return None

    df = pd.read_excel(path)
    df = df.dropna(subset=["CustomerID"])
    df = df[df["Quantity"] > 0]
    df = df[df["UnitPrice"] > 0]
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
    return df


df = load_data(DATA_PATH)
if df is None or df.empty:
    st.error(f"Raw data file not found or no valid rows: {DATA_PATH.resolve()}")
    st.write("Please place the raw Excel file at `data/raw/online_retail.xlsx` relative to this notebook.")
    st.stop()

# -------------------------------
# SIDEBAR FILTERS
# -------------------------------
st.sidebar.header("Filters")

countries = sorted(df["Country"].dropna().unique())
selected_countries = st.sidebar.multiselect(
    "Select Country",
    countries,
    default=countries,
)

if selected_countries:
    df = df[df["Country"].isin(selected_countries)]

# -------------------------------
# KPIs
# -------------------------------
col1, col2, col3 = st.columns(3)
col1.metric("Total Revenue", f"{df['TotalPrice'].sum():,.0f}")
col2.metric("Total Orders", df['InvoiceNo'].nunique())
col3.metric("Total Customers", df['CustomerID'].nunique())

# -------------------------------
# SALES TREND
# -------------------------------
st.subheader("📈 Sales Trend")
weekly_sales = df.groupby("InvoiceDate")["TotalPrice"].sum()
fig1 = plt.figure(figsize=(10, 4))
weekly_sales.plot()
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.tight_layout()
st.pyplot(fig1)

# -------------------------------
# TOP PRODUCTS
# -------------------------------
st.subheader("🏆 Top Products")

top_products = df.groupby("Description")["TotalPrice"].sum().sort_values(ascending=False).head(10)
fig2 = plt.figure(figsize=(10, 4))
top_products.plot(kind="bar")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Revenue")
plt.tight_layout()
st.pyplot(fig2)

# -------------------------------
# CUSTOMER SEGMENTATION
# -------------------------------
st.subheader("👥 Customer Segmentation")

snapshot_date = df["InvoiceDate"].max()
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,
    "InvoiceNo": "count",
    "TotalPrice": "sum",
})
rfm.columns = ["Recency", "Frequency", "Monetary"]

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

kmeans = KMeans(n_clusters=4, random_state=42)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

fig3 = plt.figure(figsize=(10, 5))
sns.scatterplot(x="Recency", y="Monetary", hue="Cluster", data=rfm, palette="tab10")
plt.tight_layout()
st.pyplot(fig3)

# -------------------------------
# DEMAND FORECASTING
# -------------------------------
st.subheader("🔮 Demand Forecast (Next 30 Days)")

daily_sales_df = df.groupby("InvoiceDate")["TotalPrice"].sum().reset_index()
daily_sales_df.columns = ["ds", "y"]

model = Prophet()
model.fit(daily_sales_df)

future = model.make_future_dataframe(periods=30)
forecast = model.predict(future)

fig4 = model.plot(forecast)
plt.tight_layout()
st.pyplot(fig4)

# -------------------------------
# CHURN PREDICTION
# -------------------------------
st.subheader("⚠️ Churn Prediction")

last_purchase = df.groupby("CustomerID")["InvoiceDate"].max()
churn = (snapshot_date - last_purchase).dt.days > 90
churn_df = churn.reset_index()
churn_df.columns = ["CustomerID", "Churn"]

fig5 = plt.figure(figsize=(6, 4))
churn_df["Churn"].value_counts().plot(kind="bar")
plt.title("Churn vs Active Customers")
plt.xlabel("Churn")
plt.ylabel("Count")
plt.tight_layout()
st.pyplot(fig5)

st.write(churn_df.head())

# -------------------------------
# INVENTORY RECOMMENDATION
# -------------------------------
st.subheader("📦 Inventory Recommendation")

forecast_30 = forecast[["ds", "yhat"]].tail(30)
stock = forecast_30["yhat"].sum()
st.metric("Recommended Stock (Next 30 Days)", f"{stock:,.0f}")

# -------------------------------
# DOWNLOAD OPTION
# -------------------------------
st.download_button(
    "Download Processed Data",
    data=df.to_csv(index=False),
    file_name="processed_data.csv",
    mime="text/csv",
)
